# Brand Manager in a Box: add the playbook (RAG)

**Before you start:** check the top right says A100, then Runtime > **Run all**.

Takes about 1 to 1.5 hours. Keep this tab open. Results save to Drive `BrandManagerAI/rag_answers.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/BrandManagerAI'
import os; print(sorted(os.listdir(DIR)))

In [ ]:
%%capture
!pip install -U unsloth sentence-transformers
!pip install 'transformers==5.10.2'

## 1. Build the search over the playbook
Each passage becomes a list of numbers that captures its meaning. A question is turned into numbers the same way, and the closest passages are the best matches.

In [ ]:
import json, numpy as np, torch
from sentence_transformers import SentenceTransformer
passages = [json.loads(l) for l in open(f'{DIR}/passages.jsonl')]
embedder = SentenceTransformer('BAAI/bge-base-en-v1.5', device = 'cuda')
emb_path = f'{DIR}/passage_embeddings.npy'
if os.path.exists(emb_path) and np.load(emb_path).shape[0] == len(passages):
    P = np.load(emb_path)
else:
    P = embedder.encode([p['text'] for p in passages], batch_size = 128, normalize_embeddings = True,
                        show_progress_bar = True)
    np.save(emb_path, P)
P = torch.tensor(P, device = 'cuda')
QUERY = 'Represent this sentence for searching relevant passages: '
def lookup(question, k = 4):
    q = embedder.encode([QUERY + question], normalize_embeddings = True, convert_to_tensor = True)
    top = torch.topk(P @ q[0], k).indices.tolist()
    return [passages[i] for i in top]
for p in lookup('How should I price a premium product above the category leader?'):
    print(p['source'], '|', p['text'][:200], '\n')

## 2. Load the trained bot (the add-on from the training run)

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
model, tokenizer = FastModel.from_pretrained(
    model_name = f'{DIR}/brand-manager-lora',
    max_seq_length = 4096,
    load_in_4bit = False,
)
tokenizer = get_chat_template(tokenizer, chat_template = 'gemma-4')
FastModel.for_inference(model)

## 3. Answer the 40 test questions, with the playbook open

In [ ]:
SYSTEM = json.loads(open(f'{DIR}/test.jsonl').readline())['messages'][0]['content']
def with_notes(question):
    notes = '\n\n'.join(f'[{i + 1}] ' + p['text'] for i, p in enumerate(lookup(question)))
    return (question + '\n\n---\nNotes from your brand playbook. Use whatever helps and ignore the rest. '
            'Never name the companies, brands, programs or people that appear in the notes.\n\n' + notes)
tok = getattr(tokenizer, 'tokenizer', tokenizer)
tok.padding_side = 'left'
def clean(text):
    return text.strip().removeprefix('thought').strip()  # empty thinking-channel marker
def answer_batch(questions, max_new_tokens = 500):
    convos = [[{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': with_notes(q)}] for q in questions]
    inputs = tokenizer.apply_chat_template(convos, add_generation_prompt = True, tokenize = True, padding = True,
                                           return_dict = True, return_tensors = 'pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, use_cache = True,
                         temperature = 1.0, top_p = 0.95, top_k = 64)
    n = inputs['input_ids'].shape[1]
    return [clean(tokenizer.decode(o[n:], skip_special_tokens = True)) for o in out]
def answer_all(questions, size = 8):
    results = []
    for i in range(0, len(questions), size):
        results += answer_batch(questions[i:i + size])
        print(f'{len(results)} / {len(questions)} done')
    return results
rows = json.load(open(f'{DIR}/head_to_head.json'))
questions = [r['request'] for r in rows]
tuned_rag = answer_all(questions)
with model.disable_adapter():
    base_rag = answer_all(questions)
for r, t, b in zip(rows, tuned_rag, base_rag):
    r['tuned_rag'], r['base_rag'] = t, b
json.dump(rows, open(f'{DIR}/rag_answers.json', 'w'), indent = 2)
print('saved', len(rows))
print('\nQUESTION:', rows[0]['request'][:300], '\n\nTRAINED + PLAYBOOK:', tuned_rag[0][:900])

## 4. Try it yourself

In [ ]:
question = 'We are launching a premium dog food at twice the price of the category leader. How do we justify the price?'
print(answer_batch([question])[0])

## 5. Switch off the GPU
When you've finished: Runtime > **Disconnect and delete runtime** > Yes.